# CGM paper post-processing — one run at a time

This notebook is the analysis entry point. Set `DIMENSION` and `RUN_FOLDER` once in Section 1. All outputs for that result set are written under `figures_paper/<dimension>/<experiment>/`.

## 1. Setup and selected result set

In [ ]:
from __future__ import annotations

import argparse
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import multivariate_normal, norm

# A notebook has no __file__. Resolve the project root robustly whether the
# notebook is launched from <PROJECT_ROOT>/ or <PROJECT_ROOT>/Analysis/.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / "Analysis").is_dir() else CWD.parent
if not (PROJECT_ROOT / "Analysis").is_dir():
    raise RuntimeError("Open this notebook from the project root or Analysis/")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Analysis.paper_config import (
    ARM_COLORS,
    ARM_NAMES,
    ANALYSIS_WINDOW_DEFINITION,
    COMPONENT_COLORS,
    CONTROL_IDS,
    DISPLAY_PATIENT_IDS,
    FIGURES_ROOT,
    INFERENCE_SEED,
    PAPER_BAND_PCT,
    PAPER_INTERVAL,
    PAPER_N_GRID,
    PARTICIPANTS_FILE,
    PEDAP_DATA_DIR,
    RESULTS_ROOT,
    TREATMENT_IDS,
    WASSERSTEIN_SEED,
    dataset_fingerprint,
    treatment_comparison_supported,
)
from Analysis.paper_config import CGM_FILE as PEDAP_CGM_FILE
from Analysis.paper_config import N_NOMINAL_WEEKS

# -----------------------------------------------------------------------------
# SELECT EXACTLY ONE RESULT SET HERE.
# -----------------------------------------------------------------------------
DIMENSION = "d2"                 # "d1" or "d2"
RUN_FOLDER = RESULTS_ROOT / ("run14" if DIMENSION == "d1" else "run15")

if DIMENSION not in {"d1", "d2"}:
    raise ValueError("DIMENSION must be 'd1' or 'd2'")
RUN_FOLDER = Path(RUN_FOLDER).resolve()
RUN_NAME = RUN_FOLDER.name
EXPECTED_DIMENSION = 1 if DIMENSION == "d1" else 2

from Analysis.clinical_support import validate_rct_run

RUN_PARAMS = validate_rct_run(RUN_FOLDER)
DATASET_FINGERPRINT = RUN_PARAMS["dataset_fingerprint"]
OUTPUT_FOLDER = FIGURES_ROOT / DIMENSION / RUN_NAME

from Analysis.paper_config import pedap_manifest as _pedap_manifest
_MANIFEST = _pedap_manifest(PEDAP_DATA_DIR)
PARTICIPANTS_FILE = PEDAP_DATA_DIR / _MANIFEST["files"]["participants"]
PEDAP_CGM_FILE = PEDAP_DATA_DIR / _MANIFEST["files"]["cgm"]

INTERVAL = PAPER_INTERVAL
N_GRID = PAPER_N_GRID
BAND_PCT = PAPER_BAND_PCT
ROSENBLATT_IDS = DISPLAY_PATIENT_IDS

N_BOOTSTRAP = 9999
N_PERMUTATIONS = 9999
WASSERSTEIN_SAMPLES = 250
WASSERSTEIN_SUPPORT = 250
# Individual trajectories are drawn on CLINICAL time (weeks since
# randomization).  Set this to True only to reproduce a historical figure.
LEGACY_INDIVIDUAL_NORMALIZED_TIME = False

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT        :", PROJECT_ROOT)
print("DIMENSION           :", DIMENSION)
print("RUN_FOLDER          :", RUN_FOLDER)
print("OUTPUT_FOLDER       :", OUTPUT_FOLDER)
print("PEDAP_DATA_DIR      :", PEDAP_DATA_DIR)
print("DATASET FINGERPRINT :", DATASET_FINGERPRINT)
print()
print("PEDAP RCT: available-case descriptive curves; fixed-cohort inference")
print()
print("NOMINAL RCT WEEKS   :", N_NOMINAL_WEEKS)
print("PAPER_INTERVAL      :", INTERVAL)
if not treatment_comparison_supported():
    print(
        "\n*** WARNING: this PEDAP dataset is flagged "
        "treatment_comparison_supported=false.\n"
        "*** RCT CGM coverage is not comparable between arms; no CLC-vs-SC\n"
        "*** conclusion may be drawn from any figure in this notebook. ***\n"
    )


## 2. Import analysis modules

In [ ]:
from Analysis.paper_figures import generate_paper_figures
from Analysis.wild_bootstrap_mmd import run_global_functional_mmd_analyses
import Analysis.rosenblatt_diagnostics as rosenblatt_module

from Visualization.plots import _get_patient_dir, _load_patient_results
from load import load_patient_weeks

run_rosenblatt_diagnostics = rosenblatt_module.run_diagnostics

print(f"Control n={len(CONTROL_IDS)}, Treatment n={len(TREATMENT_IDS)}")


def _resolve_display_ids(candidates, run_folder, seed):
    """Replace any candidate absent from this run's cohort with another
    patient drawn at random (reproducibly) from the same run, keeping the
    requested count. A patient "works" here if it has a result folder and a
    result_fit.json -- the same minimum the Rosenblatt and individual-panel
    cells below already require before touching a patient. Not every run
    shares the same cohort (e.g. a patient excluded upstream for one run may
    still be requested by DISPLAY_PATIENT_IDS), so this is resolved per run
    rather than assumed fixed.
    """

    def _patient_id(path):
        name = path.name
        return int(name.split("_", 1)[1] if name.startswith("patient_") else name.split("=", 1)[1])

    def _is_available(patient_id):
        patient_dir = _get_patient_dir(str(run_folder), patient_id)
        return patient_dir is not None and (Path(patient_dir) / "result_fit.json").exists()

    resolved = [pid for pid in candidates if _is_available(pid)]
    if len(resolved) < len(candidates):
        cohort_ids = sorted({
            _patient_id(path) for pattern in ("patient_*", "ids=*")
            for path in run_folder.glob(pattern)
            if (path / "result_fit.json").exists()
        })
        pool = [pid for pid in cohort_ids if pid not in resolved]
        np.random.default_rng(seed).shuffle(pool)
        resolved += pool[: len(candidates) - len(resolved)]
    return tuple(resolved)


ROSENBLATT_IDS = _resolve_display_ids(DISPLAY_PATIENT_IDS, RUN_FOLDER, INFERENCE_SEED)
print("Requested display IDs:", DISPLAY_PATIENT_IDS)
print("Resolved display IDs for this run:", ROSENBLATT_IDS)


## 3. Pre-flight audit

In [ ]:
required_paths = {
    "selected run": RUN_FOLDER,
    "PEDAP participants": PARTICIPANTS_FILE,
    "PEDAP CGM": PEDAP_CGM_FILE,
}
for name, path in required_paths.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{name:12s} {status:8s} {path}")

if not RUN_FOLDER.is_dir():
    raise FileNotFoundError(f"Selected result folder does not exist: {RUN_FOLDER}")

dictionary_path = RUN_FOLDER / "global_dictionary.json"
if not dictionary_path.exists():
    raise FileNotFoundError(f"Missing global dictionary: {dictionary_path}")
with dictionary_path.open("r", encoding="utf-8") as handle:
    dictionary_metadata = json.load(handle)
actual_dimension = int(dictionary_metadata.get("d", -1))
if actual_dimension != EXPECTED_DIMENSION:
    raise RuntimeError(
        f"DIMENSION={DIMENSION} expects d={EXPECTED_DIMENSION}, but selected run has d={actual_dimension}"
    )

patient_dirs = list(RUN_FOLDER.glob("patient_*")) + list(RUN_FOLDER.glob("ids=*"))
print(f"Patient-result folders: {len(patient_dirs)}")
run_fingerprint = dictionary_metadata.get("dataset_fingerprint")
if run_fingerprint != DATASET_FINGERPRINT:
    raise RuntimeError(
        f"Selected run was fitted on PEDAP dataset {run_fingerprint!r}, but its "
        f"primary dataset is {DATASET_FINGERPRINT!r}. "
        "Clinical results are never mixed across datasets."
    )
print(f"Analysis window: {ANALYSIS_WINDOW_DEFINITION}")
print(
    f"{N_NOMINAL_WEEKS} nominal RCT weeks; each participant contributes the "
    "usable weekly distributions it actually has, at their real block midpoints."
)


## 4. Cohort-level manuscript figures

In [ ]:
print(f"[paper] {DIMENSION}: {RUN_FOLDER} -> {OUTPUT_FOLDER}")
paper_metadata = generate_paper_figures(
    TREATMENT_IDS,
    CONTROL_IDS,
    RUN_FOLDER,
    interval=INTERVAL,
    n_grid=N_GRID,
    band_pct=BAND_PCT,
    output_folder=OUTPUT_FOLDER,
    expected_dimension=EXPECTED_DIMENSION,
)
print("included:", paper_metadata["included_counts"])
print("coverage:", json.dumps(paper_metadata["coverage_report"], indent=2))


### 4.1. Temporal evolution of the global GMM mean and covariance

For each patient, use the fixed MMD Gaussian dictionary and the time-dependent NODE component weights:

$$m(t)=\sum_k w_k(t)\mu_k,\qquad C(t)=\sum_k w_k(t)\left[\Sigma_k+(\mu_k-m(t))(\mu_k-m(t))^\top\right].$$

Here “global” means the complete Gaussian mixture for one patient. Parameters are in original units. For d=2, plot both mean coordinates and all three distinct covariance entries (including the cross-covariance). Thin curves represent patients; thick curves are the arithmetic mean of the **patient moments** within each arm, not the covariance of a pooled cohort distribution. Separate panels are also saved for the selected display patients. The time axis uses the original fitted study-week coordinates.


In [ ]:
from Analysis.paper_figures import _display_time, _load_interval_data, _save_figure


def global_mixture_moments(weights, means, covs):
    """Exact mixture moments; weights have shape (..., K), dictionary (K,d)/(K,d,d)."""
    weights = np.asarray(weights, dtype=float)
    means = np.asarray(means, dtype=float)
    covs = np.asarray(covs, dtype=float)
    if means.ndim != 2 or covs.shape != (means.shape[0], means.shape[1], means.shape[1]):
        raise ValueError("Expected dictionary shapes (K,d) and (K,d,d)")
    if weights.ndim < 2 or weights.shape[-1] != means.shape[0]:
        raise ValueError("NODE weights must have shape (..., K)")
    if not all(np.isfinite(x).all() for x in (means, covs)):
        raise ValueError("Non-finite dictionary parameters")
    # Under available_case a participant is all-NaN outside its own support:
    # those cells stay NaN and are skipped by the nan-aware reductions below.
    covered = np.isfinite(weights).all(axis=-1)
    present = weights[covered]
    if present.size:
        if np.any(present < -1e-7) or not np.allclose(
            present.sum(axis=-1), 1.0, atol=1e-6, rtol=0
        ):
            raise ValueError("NODE weights must be nonnegative and sum to one")
    # Remove only floating-point simplex residuals.
    weights = np.maximum(weights, 0.0)
    weights = weights / weights.sum(axis=-1, keepdims=True)
    mean = np.einsum("...k,kd->...d", weights, means)
    centered = means - mean[..., None, :]
    covariance = np.einsum("...k,kij->...ij", weights, covs)
    covariance += np.einsum("...k,...ki,...kj->...ij", weights, centered, centered)
    return mean, covariance


moment_data = _load_interval_data(
    TREATMENT_IDS, CONTROL_IDS, str(RUN_FOLDER), interval=INTERVAL, n_grid=N_GRID,
)
# Participants' fitted NODE supports genuinely differ under PEDAP v2 (a
# truncated first week, a real 13-week visit date, dropouts, missing weeks):
# there is no single shared study-week support to require or reconstruct from
# one participant.  _load_interval_data already interpolated every included
# participant's trajectory onto ``common_grid`` -- the intersection of their
# real supports, in weeks since randomization -- so that is the correct time
# axis to plot moments on, unmodified.
moment_times = _display_time(moment_data["common_grid"])
_avail = moment_data.get("n_available_per_grid_point")
if _avail is not None:
    print(f"[moments] grid mode: {moment_data['common_grid_mode']} | "
          f"span {moment_times[0]:.2f}-{moment_times[-1]:.2f} weeks | "
          f"participants per point: min {int(_avail.min())}, max {int(_avail.max())} "
          f"of {len(moment_data['included_ids'])}")
global_means, global_covariances = global_mixture_moments(
    moment_data["trajectories"],
    moment_data["means_original_units"],
    moment_data["covs_original_units"],
)
if global_means.shape[-1] != EXPECTED_DIMENSION:
    raise ValueError("Moment dimension does not match DIMENSION")

mean_labels = [r"$m_1(t)$ [mg/dL]", r"$m_2(t)$ [mg/dL/time]"][:EXPECTED_DIMENSION]
cov_entries = [(i, j) for i in range(EXPECTED_DIMENSION) for j in range(i, EXPECTED_DIMENSION)]
cov_labels = {
    (0, 0): r"$C_{11}(t)$ [(mg/dL)$^2$]",
    (0, 1): r"$C_{12}(t)$ [(mg/dL)$^2$/time]",
    (1, 1): r"$C_{22}(t)$ [(mg/dL/time)$^2$]",
}
moment_output = OUTPUT_FOLDER / "global_moments"
moment_output.mkdir(parents=True, exist_ok=True)
moment_files = []


def plot_global_moments(values, axis_labels, stem, patient_index=None):
    fig, axes = plt.subplots(
        1, len(axis_labels), figsize=(5 * len(axis_labels), 3.8),
        sharex=True, squeeze=False, layout="constrained",
    )
    labels = np.asarray(moment_data["labels"])
    for j, ax in enumerate(axes[0]):
        if patient_index is None:
            for arm in (0, 1):
                curves = values[labels == arm, :, j]
                ax.plot(moment_times, curves.T, color=ARM_COLORS[arm], alpha=0.12, lw=0.7)
                # nan-aware: each time point averages only the participants
                # whose own support covers it (available-case).
                ax.plot(moment_times, np.nanmean(curves, axis=0), color=ARM_COLORS[arm],
                        lw=2.4, label=ARM_NAMES[arm])
        else:
            arm = int(labels[patient_index])
            ax.plot(moment_times, values[patient_index, :, j], color=ARM_COLORS[arm],
                    lw=2.2, label=f"Patient {moment_data['included_ids'][patient_index]} ({ARM_NAMES[arm]})")
        ax.set(xlabel="Study week", ylabel=axis_labels[j],
               xlim=(moment_times[0], moment_times[-1]))
        ax.grid(alpha=0.2, linestyle="--", linewidth=0.6)
    axes[0, 0].legend(frameon=False)
    display(fig)
    return _save_figure(fig, moment_output, stem)


covariance_curves = np.stack([global_covariances[..., i, j] for i, j in cov_entries], axis=-1)
for name, values, axis_labels in (
    ("mean", global_means, mean_labels),
    ("covariance", covariance_curves, [cov_labels[entry] for entry in cov_entries]),
):
    moment_files += plot_global_moments(values, axis_labels, f"global_{name}_by_arm")
    for patient_id in ROSENBLATT_IDS:
        if patient_id in moment_data["included_ids"]:
            index = moment_data["included_ids"].index(patient_id)
            moment_files += plot_global_moments(
                values, axis_labels, f"global_{name}_patient_{patient_id}", patient_index=index,
            )
        else:
            print(f"[global moments] patient {patient_id}: excluded by the cohort loader")

# Save the exact plotted patient trajectories, including the full symmetric covariance.
np.savez_compressed(
    moment_output / "global_moments.npz", study_weeks=moment_times,
    patient_ids=moment_data["included_ids"], arm_labels=moment_data["labels"],
    means=global_means, covariances=global_covariances,
)
print(f"[global moments] saved {len(moment_files)} figure files -> {moment_output}")


### 4.2. Component cross-covariances and Treatment − Control moment curves

For d=2 the requested within-Gaussian term is $a_k(t)=w_k(t)\Sigma_{k,12}$. Its sum is the within-component part of the mixture covariance. The full component contribution is $b_k(t)=w_k(t)[\Sigma_{k,12}+(\mu_{k,1}-m_1(t))(\mu_{k,2}-m_2(t))]$, and $\sum_k b_k(t)=C_{12}(t)$. Both are plotted separately, using the same patient and arm conventions as Section 4.1.

For each mean coordinate and each distinct covariance entry, plot the arithmetic mean of the patient moments in Treatment minus that in Control, at every fitted time. Covariances are computed per patient before averaging; they are not covariances of a pooled group mixture. No extrapolation beyond the saved NODE support is used.


In [ ]:
# Difference of arm-average patient moments, in the original measurement units.
moment_labels = np.asarray(moment_data["labels"])
moment_difference_files = []
moment_difference_arrays = {}
for name, values, axis_labels in (
    ("mean", global_means, mean_labels),
    ("covariance", covariance_curves, [cov_labels[entry] for entry in cov_entries]),
):
    difference = (np.nanmean(values[moment_labels == 1], axis=0)
                  - np.nanmean(values[moment_labels == 0], axis=0))
    moment_difference_arrays[name] = difference
    fig, axes = plt.subplots(1, len(axis_labels), figsize=(5 * len(axis_labels), 3.8),
                             sharex=True, squeeze=False, layout="constrained")
    for j, ax in enumerate(axes[0]):
        ax.plot(moment_times, difference[:, j], color="#553c7b", lw=2.2)
        ax.axhline(0, color="0.4", lw=0.9, linestyle="--")
        ax.set(xlabel="Study week", ylabel="Treatment − Control: " + axis_labels[j],
               xlim=(moment_times[0], moment_times[-1]))
        ax.grid(alpha=0.18)
    display(fig)
    moment_difference_files += _save_figure(fig, moment_output, f"global_{name}_treatment_minus_control")

np.savez_compressed(
    moment_output / "global_moment_group_differences.npz", study_weeks=moment_times,
    mean_difference=moment_difference_arrays["mean"],
    covariance_difference=np.nanmean(global_covariances[moment_labels == 1], axis=0)
                          - np.nanmean(global_covariances[moment_labels == 0], axis=0),
)

component_covariance_files = []
if EXPECTED_DIMENSION == 2:
    dictionary_means = np.asarray(moment_data["means_original_units"])
    dictionary_covs = np.asarray(moment_data["covs_original_units"])
    weights = np.asarray(moment_data["trajectories"], dtype=float)
    weights = np.maximum(weights, 0.0)
    weights /= weights.sum(axis=-1, keepdims=True)
    component_cross_covariance = weights * dictionary_covs[:, 0, 1]
    centered_means = dictionary_means[None, None, :, :] - global_means[:, :, None, :]
    component_total_cross_covariance = component_cross_covariance + (
        weights * centered_means[..., 0] * centered_means[..., 1]
    )
    np.testing.assert_allclose(component_total_cross_covariance.sum(axis=-1),
                               global_covariances[..., 0, 1], atol=1e-9)
    for name, values, symbol in (
        ("within", component_cross_covariance, "a"),
        ("total", component_total_cross_covariance, "b"),
    ):
        labels = [rf"${symbol}_{k+1}(t)$ [(mg/dL)$^2$/time]" for k in range(weights.shape[-1])]
        component_covariance_files += plot_global_moments(
            values, labels, f"component_cross_covariance_{name}_by_arm",
        )
        for patient_id in ROSENBLATT_IDS:
            if patient_id in moment_data["included_ids"]:
                component_covariance_files += plot_global_moments(
                    values, labels, f"component_cross_covariance_{name}_patient_{patient_id}",
                    patient_index=moment_data["included_ids"].index(patient_id),
                )
    np.savez_compressed(
        moment_output / "component_cross_covariances.npz", study_weeks=moment_times,
        patient_ids=moment_data["included_ids"], arm_labels=moment_labels,
        weighted_component_covariances=component_cross_covariance,
        full_component_contributions=component_total_cross_covariance,
    )
else:
    print("[component cross-covariances] d=1 has no off-diagonal entry")
print(f"[moment comparisons] saved {len(moment_difference_files) + len(component_covariance_files)} figure files")


### 4.3. Descriptive pre-randomization baseline to pooled follow-up

This scalar benchmark compares the 14-day pre-randomization baseline with
all CGM readings in the randomized follow-up window. Both sides require at
least 168 hours; summaries use participants with both sides available.
The unadjusted between-arm comparison of paired changes is descriptive,
separate from the primary functional MMD tests.


In [ ]:
from Analysis.wadwa_endpoints import generate as generate_baseline_endpoint

endpoint_result = generate_baseline_endpoint(
    FIGURES_ROOT / "baseline_endpoint",
)
display(pd.DataFrame(endpoint_result["summary"]))
print(endpoint_result["between_arm_change"])


## 5. Primary functional wild-bootstrap MMD + exploratory diagnostics

In [ ]:
def save_wild_null_distribution(result: dict, output_stem: Path, estimand: str):
    """Paper-style null distribution. Statistical code stays in wild_bootstrap_mmd.py."""
    observed = float(result["scaled_mmd_v_statistic"])
    bootstrap = np.asarray(result["wild_bootstrap"]["_replicates"], dtype=float)
    permutation = result.get("permutation_validation")

    ncols = 2 if permutation is not None else 1
    fig, axes = plt.subplots(
        1, ncols,
        figsize=(11.5 if ncols == 2 else 6.0, 4.2),
    )
    axes = np.atleast_1d(axes)

    ax = axes[0]
    bins = min(55, max(20, int(np.sqrt(len(bootstrap)))))
    ax.hist(
        bootstrap, bins=bins, density=True,
        color="#d9e6ee", edgecolor="white", linewidth=0.5,
    )
    ax.axvline(observed, color="#222222", linewidth=2.1)
    ax.set_xlabel("Scaled MMD V-statistic")
    ax.set_ylabel("Density")
    ax.set_title("Wild bootstrap")
    ax.grid(alpha=0.15, linestyle="--", linewidth=0.6)
    ax.text(
        0.97, 0.95,
        f"$p={result['wild_bootstrap']['pvalue']:.4g}$",
        transform=ax.transAxes,
        ha="right", va="top", fontsize=11,
    )

    if permutation is not None:
        permuted = np.asarray(permutation["_replicates"], dtype=float)
        ax = axes[1]
        bins = min(55, max(20, int(np.sqrt(len(permuted)))))
        ax.hist(
            permuted, bins=bins, density=True,
            color="#efe2d3", edgecolor="white", linewidth=0.5,
        )
        ax.axvline(observed, color="#222222", linewidth=2.1)
        ax.set_xlabel("Scaled MMD V-statistic")
        ax.set_ylabel("Density")
        ax.set_title("Permutation validation")
        ax.grid(alpha=0.15, linestyle="--", linewidth=0.6)
        ax.text(
            0.97, 0.95,
            f"$p={permutation['pvalue']:.4g}$",
            transform=ax.transAxes,
            ha="right", va="top", fontsize=11,
        )

    # Identify the pre-randomization anchor explicitly.
    _ESTIMAND_TITLES = {
        "absolute": "Absolute trajectories",
        "change_from_baseline": "Change from pre-randomization baseline",
    }
    fig.suptitle(
        _ESTIMAND_TITLES.get(estimand, estimand),
        fontsize=12,
        y=1.01,
    )
    fig.tight_layout()
    output_stem = Path(output_stem)
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    saved = []
    for suffix, dpi in ((".pdf", None), (".png", 300)):
        path = output_stem.with_suffix(suffix)
        fig.savefig(path, dpi=dpi, bbox_inches="tight", pad_inches=0.03)
        saved.append(path)
    plt.close(fig)
    return saved



wild_output = OUTPUT_FOLDER / "wild_bootstrap"
wild_results = run_global_functional_mmd_analyses(
    RUN_FOLDER,
    interval=INTERVAL,
    n_grid=N_GRID,
    n_bootstrap=N_BOOTSTRAP,
    n_permutations=N_PERMUTATIONS,
    seed=INFERENCE_SEED,
    output_dir=wild_output,
)

for estimand in wild_results:
    result = wild_results[estimand]["global"]
    save_wild_null_distribution(
        result,
        wild_output / f"wild_bootstrap_{estimand}_null",
        estimand,
    )
    perm = result.get("permutation_validation") or {}
    anchor = wild_results[estimand].get("baseline_anchor")
    note = ""
    if anchor:
        note = (f"   [pre-randomization anchor, n={anchor['n_with_baseline']}"
                f" (CLC {anchor['n_with_baseline_clc']}, "
                f"SC {anchor['n_with_baseline_sc']})]")
    print(
        f"{estimand:22s} MMD²={result['mmd_squared']:.6g}  "
        f"wild p={result['wild_bootstrap']['pvalue']:.6g}  "
        f"perm p={perm.get('pvalue', float('nan')):.6g}{note}"
    )

In [ ]:
# Cumulative ACAT curves for the change-from-baseline estimand.
# Weekly: the week-centre p-values already computed by wild_bootstrap_mmd.py.
# Fine grid: the same scalar wild MMD test at every point of the N_GRID paper grid.
from Analysis.paper_figures import _load_interval_data, _save_figure
from Analysis.wild_bootstrap_mmd import cumulative_acat, pointwise_tests

ACAT_STANDALONE_TAUS_DAYS = (10.0, 14.0)
ACAT_SWEEP_TAUS_DAYS = (1.0, 3.0, 5.0, 7.0, 10.0, 14.0)


def save_cumulative_acat_figure(times, curves, stem, markers):
    """One row per (row label, p-value matrix), one panel per component; no legend."""
    n_rows, n_components = len(curves), curves[0][1].shape[1]
    fig, axes = plt.subplots(
        n_rows, n_components,
        figsize=(2.4 * n_components, 2.6 if n_rows == 1 else 1.75 * n_rows),
        sharex=True, sharey=True, squeeze=False, layout="constrained",
    )
    style = {"marker": "o", "markersize": 3.0} if markers else {}
    for row, (label, pvalues) in enumerate(curves):
        for k in range(n_components):
            ax = axes[row, k]
            ax.plot(
                times, pvalues[:, k],
                color=COMPONENT_COLORS[k], linewidth=1.5, clip_on=False, **style,
            )
            ax.axhline(0.05, color="black", linestyle="--", linewidth=0.9)
            ax.set_yscale("log")
            ax.grid(alpha=0.15, linestyle="--", linewidth=0.6)
            if row == 0:
                ax.set_title(f"Component {k + 1}", fontsize=10)
        if label is not None:
            axes[row, 0].set_ylabel(label, fontsize=9)

    smallest = min(np.nanmin(pvalues) for _, pvalues in curves)
    axes[0, 0].set_ylim(min(1e-3, 10 ** np.floor(np.log10(smallest))), 1.0)
    # The axis spans the clinical times actually tested, not a nominal 0..13.
    # Fixed nominal trial span, matching the weekly p-value panels: the tested
    # centres no longer start at week 0 once the complete-case inference window
    # does not, and a data-driven axis would hide that.
    axes[0, 0].set_xlim(0.0, float(N_NOMINAL_WEEKS))
    fig.supxlabel("Weeks since randomization", fontsize=10)
    if n_rows == 1:
        axes[0, 0].set_ylabel("Cumulative ACAT p-value")
    else:
        fig.supylabel("Cumulative ACAT p-value", fontsize=10)
    return _save_figure(fig, wild_output, stem)


# The primary change estimand is the baseline-anchored one where it exists;
CHANGE_ESTIMAND = "change_from_baseline"
print(f"[ACAT] change estimand in use: {CHANGE_ESTIMAND}")
change_weekly = wild_results[CHANGE_ESTIMAND]["weekly_pointwise"]
weekly_times = np.asarray(change_weekly["week_centers"], dtype=float)
weekly_pvalues = np.asarray(
    [
        np.nan if row["pointwise_pvalue"] is None else row["pointwise_pvalue"]
        for row in change_weekly["rows"]
    ],
    dtype=float,
).reshape(len(weekly_times), -1)

# Inference is complete-case, matching wild_bootstrap_mmd.py: the wild
# bootstrap's subject-level multipliers are not defined under time-varying
# membership.  The window and the cohort were chosen TOGETHER there (widest
# window still backed by the required share of the cohort), so this fine grid
# must reuse exactly that choice rather than re-deriving the full intersection,
# which one or two extreme participants would shrink to a mid-trial slice.
_wild_config = wild_results[CHANGE_ESTIMAND]["configuration"]
_window = _wild_config["inference_window_selection"]
_inference_ids = set(wild_results[CHANGE_ESTIMAND]["population"]["included_ids"])
print(f"[fine grid] reusing inference window {_window['window_weeks']} "
      f"(span {_window['window_span_weeks']:.2f} weeks), "
      f"n={_window['n_retained']}/{_window['n_total']}")
fine = _load_interval_data(
    tuple(pid for pid in TREATMENT_IDS if pid in _inference_ids),
    tuple(pid for pid in CONTROL_IDS if pid in _inference_ids),
    str(RUN_FOLDER),
    interval=tuple(_window["window_weeks"]), n_grid=N_GRID,
    common_grid_mode="intersection",
    expected_fingerprint=DATASET_FINGERPRINT,
)
if set(fine["included_ids"]) != _inference_ids:
    raise RuntimeError("fine-grid and weekly analyses must use the same cohort")
# fine["common_grid"] and weekly_times (wild_bootstrap_mmd.py's week_centers)
# are both already expressed in the same units -- real weeks since
# randomization, restricted to the participants' common observed support --
# so no support-equality check or rescaling is needed or valid: participants'
# real fitted supports differ under PEDAP v2 (truncated/extended weeks,
# dropouts), and forcing them to match one fixed pair of week centres would
# reintroduce the "every patient has the same 13 weeks" assumption the v2
# loader was built to remove.
fine_times = np.asarray(fine["common_grid"], dtype=float)
fine_change = np.asarray(fine["trajectories"], dtype=float)
# The fine grid MUST use the same anchor as the weekly analysis it is compared
# against, or the two curves would answer different questions.
from Analysis.baseline_projection import baseline_weights_for_run

_anchor_weights, _, _ = baseline_weights_for_run(
    RUN_FOLDER, list(fine["included_ids"]), verbose=False
)
missing = [p for p in fine["included_ids"] if p not in _anchor_weights]
if missing:
    raise RuntimeError(
        f"fine grid: {len(missing)} participants in the weekly cohort have "
        "no pre-randomization baseline"
    )
_anchor = np.array(
    [_anchor_weights[p] for p in fine["included_ids"]], dtype=float
)[:, None, :]
fine_change = fine_change - _anchor
fine_pvalues = pointwise_tests(
    fine_change, fine["labels"], fine_times,
    n_bootstrap=N_BOOTSTRAP, seed=INFERENCE_SEED,
)["wild_pvalue"]

acat_figures = []
for grid_name, times, pvalues, markers in (
    ("weekly", weekly_times, weekly_pvalues, True),
    ("fine_grid", fine_times, fine_pvalues, False),
):
    for tau in ACAT_STANDALONE_TAUS_DAYS:
        acat_figures += save_cumulative_acat_figure(
            times,
            [(None, cumulative_acat(pvalues, times, tau_days=tau))],
            f"cumulative_acat_change_from_baseline_{grid_name}_tau{tau:g}",
            markers,
        )
    acat_figures += save_cumulative_acat_figure(
        times,
        [
            (
                f"τ = {tau:g} day{'' if tau == 1 else 's'}",
                cumulative_acat(pvalues, times, tau_days=tau),
            )
            for tau in ACAT_SWEEP_TAUS_DAYS
        ],
        f"cumulative_acat_change_from_baseline_{grid_name}_tau_sweep",
        markers,
    )

for path in acat_figures:
    print(path)

## 6. Rosenblatt diagnostics — d=2 only

In [ ]:
def save_rosenblatt_3d_qq_historical(curves, patient_id: int, coordinate: str, output_stem: Path):
    """Historical paper format: viridis week colours, no title/legend, compact Week axis.

    The z-axis shows the nominal RCT week the block belongs to, anchored at
    randomization.  Blocks a participant does not have are simply absent: the
    axis is never padded to 13 curves.
    Every week is plotted (n_display_times == N_NOMINAL_WEEKS below), so the
    colour map must give each of the (up to) N_NOMINAL_WEEKS curves a
    distinct colour; a continuous map does this without repeating, unlike the
    10-colour tab10 cycle.
    """
    fig = plt.figure(figsize=(9.4, 7.4))
    axis = fig.add_subplot(111, projection="3d")
    colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(curves)))
    key = coordinate.lower()
    week_numbers = [int(item["block_index"]) + 1 for item in curves]

    for color, item, week in zip(colors, curves, week_numbers):
        theoretical, discrete = rosenblatt_module.uniform_qq_coordinates(item[f"{key}_discrete"])
        _, node = rosenblatt_module.uniform_qq_coordinates(item[f"{key}_node"])
        z = np.full_like(theoretical, week)

        # Primary saved-MMD diagnostic, matching the historical coloured curves.
        axis.plot(theoretical, discrete, z, color=color, linewidth=2.0, alpha=0.95)
        # NODE-at-same-time is kept as a subtle dotted overlay; it does not change
        # the historical visual hierarchy when the two curves nearly coincide.
        axis.plot(theoretical, node, z, color=color, linewidth=0.9, linestyle=":", alpha=0.70)
        axis.plot(
            theoretical, theoretical, z,
            color="0.55", linewidth=0.8, linestyle="--", alpha=0.70,
        )

    axis.set_xlabel("Uniform quantiles", labelpad=10)
    axis.set_ylabel("Empirical quantiles", labelpad=10)
    axis.set_zlabel("Study week", labelpad=20)
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.set_zticks(week_numbers)
    axis.set_zticklabels([str(week) for week in week_numbers])
    axis.view_init(elev=24, azim=-62)
    axis.grid(True)

    # bbox_inches="tight" underestimates the rotated 3D z-label's extent and
    # crops it; reserve the right margin explicitly instead of cropping to fit.
    fig.subplots_adjust(left=0.03, right=0.82, top=0.98, bottom=0.05)
    output_stem = Path(output_stem)
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_stem.with_suffix(".pdf"))
    fig.savefig(output_stem.with_suffix(".png"), dpi=300)
    plt.close(fig)


# Plot style belongs to the notebook. The module retains the transform/statistics.
rosenblatt_module._save_3d_qq = save_rosenblatt_3d_qq_historical

if DIMENSION == "d2":
    rosenblatt_output = OUTPUT_FOLDER / "rosenblatt"
    rosenblatt_args = argparse.Namespace(
        run=RUN_FOLDER,
        pedap_data_dir=PEDAP_DATA_DIR,
        output_dir=rosenblatt_output,
        display_patient=list(ROSENBLATT_IDS),
        # Show every fitted week: with a continuous colormap (viridis, above)
        # no two weeks share a colour, unlike the 10-colour tab10 cycle this
        # replaced.
        n_display_times=N_NOMINAL_WEEKS,
        variance_floor=1e-10,
    )
    rosenblatt_results = run_rosenblatt_diagnostics(rosenblatt_args)
    print("Rosenblatt display patients:", rosenblatt_results["display_patient_ids"])
    print("Rosenblatt output:", rosenblatt_output)
else:
    rosenblatt_results = None
    print("[Rosenblatt] skipped: only defined for d=2")


## 7. Legacy-style individual-patient manuscript panels — d=2 only

In [ ]:

def arm_code_for_patient(patient_id: int) -> str:
    """The trial's own arm code (CLC / SC) from the PEDAP randomization."""
    if patient_id in TREATMENT_IDS:
        return "CLC"
    if patient_id in CONTROL_IDS:
        return "SC"
    raise ValueError(f"patient {patient_id} is not a randomized PEDAP participant")


def arm_label_for_patient(patient_id: int) -> int:
    return {"CLC": 1, "SC": 0}[arm_code_for_patient(patient_id)]


def arm_for_patient(patient_id: int) -> str:
    return ARM_NAMES[arm_label_for_patient(patient_id)]


def save_pdf_png(fig, output_stem: Path, dpi: int = 300):
    """One local saving convention for all notebook-defined patient panels."""
    output_stem = Path(output_stem)
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    paths = []
    for suffix, local_dpi in ((".pdf", None), (".png", dpi)):
        path = output_stem.with_suffix(suffix)
        fig.savefig(path, dpi=local_dpi, bbox_inches="tight", pad_inches=0.03)
        paths.append(path)
    plt.close(fig)
    return paths


def load_patient_bundle(patient_id: int, run_dir: Path):
    """Load the fit and the fit-compatible NODE without using a latest-folder fallback."""
    patient_dir = _get_patient_dir(run_dir, patient_id)
    if patient_dir is None:
        raise FileNotFoundError(f"patient {patient_id}: no result directory in {run_dir}")
    patient_dir = Path(patient_dir)

    result = _load_patient_results(patient_dir, include_covs=True)
    if result is None:
        raise FileNotFoundError(f"patient {patient_id}: no fit-compatible NODE result")

    fit_path = patient_dir / "result_fit.json"
    fit = json.loads(fit_path.read_text())
    fit_config = fit.get("config", {})
    if fit_config.get("dataset_fingerprint") != DATASET_FINGERPRINT:
        raise RuntimeError(
            f"patient {patient_id}: fitted on PEDAP dataset "
            f"{fit_config.get('dataset_fingerprint')!r}, not {DATASET_FINGERPRINT!r}"
        )

    fit_weights = np.asarray(fit["results"]["weights"], dtype=float)
    observation_times = np.asarray(fit_config["observation_times_weeks"], dtype=float)
    # A participant contributes the usable weekly distributions it has; the
    # count varies between participants and is not required to be 13.
    if len(fit_weights) != len(observation_times) or len(fit_weights) > int(N_NOMINAL_WEEKS):
        raise RuntimeError(
            f"patient {patient_id}: {len(fit_weights)} fitted weekly states "
            f"against {len(observation_times)} observation times"
        )

    trajectory = np.asarray(result["traj_weights"], dtype=float)
    if trajectory.ndim == 3 and trajectory.shape[1] == 1:
        trajectory = trajectory[:, 0, :]
    trajectory_times = np.asarray(result["trajectory_time_grid_weeks"], dtype=float)

    return {
        "patient_dir": patient_dir,
        "fit": fit,
        "fit_weights": fit_weights,
        "observation_times": observation_times,
        "trajectory": trajectory,
        "trajectory_times": trajectory_times,
        "means": np.asarray(result["means"], dtype=float),
        "covs": np.asarray(result["covs"], dtype=float),
        "means_original": np.asarray(result["means_original_units"], dtype=float),
        "covs_original": np.asarray(result["covs_original_units"], dtype=float),
        "node_config": result.get("node_config", {}),
    }


def node_weights_at_times(bundle, times):
    traj = bundle["trajectory"]
    grid = bundle["trajectory_times"]
    return np.column_stack([
        np.interp(times, grid, traj[:, k]) for k in range(traj.shape[1])
    ])


In [ ]:
def _day_series(day, gap_minutes=15.0):
    """One day's readings as (minute_of_day, glucose), gaps left unjoined."""
    day = day.sort_values("minutes")
    m = day["minutes"].to_numpy(dtype=float)
    g = day["glucose"].to_numpy(dtype=float)
    if len(m) > 1:
        breaks = np.flatnonzero(np.diff(m) > gap_minutes) + 1
        m = np.insert(m, breaks, np.nan)
        g = np.insert(g, breaks, np.nan)
    return m, g


def _patient_days(cgm: pd.DataFrame, patient_id: int):
    """One (time-of-day minutes, glucose) series per follow-up calendar day.

    The window is already the SAP follow-up window: the processed CGM table
    holds nothing outside it, so no date arithmetic is repeated here and no
    "first CGM recording" is used as a time origin.
    """
    patient = cgm[cgm["PtID"] == patient_id]
    if patient.empty:
        raise ValueError(f"patient {patient_id} has no CGM in the PEDAP follow-up window")
    minutes = (patient["DeviceDtTm"].dt.hour * 60 + patient["DeviceDtTm"].dt.minute)
    frame = pd.DataFrame({
        "day": patient["DeviceDtTm"].dt.normalize(),
        "minutes": minutes.to_numpy(),
        "glucose": patient["GlucoseMgDl"].to_numpy(dtype=float),
    })
    return [group.sort_values("minutes") for _, group in frame.groupby("day")]


def plot_patient_timesignal(patient_id: int, cgm: pd.DataFrame, output_dir: Path):
    """Daily-CGM overlay over the participant's SAP follow-up window."""
    arm = arm_for_patient(patient_id)
    color = ARM_COLORS[arm_label_for_patient(patient_id)]

    fig, ax = plt.subplots(figsize=(12, 5))
    for day in _patient_days(cgm, patient_id):
        # Gaps are real missingness and are never bridged by a drawn line.
        # Plot the real minute-of-day of each reading. A fixed 5-minute grid
        # would miss almost everything: Dexcom timestamps carry seconds of
        # jitter, so readings land on 3, 8, 13... not on 0, 5, 10. The line is
        # broken across genuine gaps instead of drawn straight through them.
        m, g = _day_series(day)
        ax.plot(m / 5.0, g, color=color, alpha=0.30, lw=0.60)

    ax.set_xlim(0, 287)
    ax.set_ylim(0, 420)
    ax.set_xlabel("Time of day (5-min slots)", fontsize=15)
    ax.set_ylabel("Glucose (mg/dL)", fontsize=15)
    ax.tick_params(labelsize=12)
    ax.grid(alpha=0.22)
    fig.tight_layout()

    stem = output_dir / f"timesignal_{arm}_patient{patient_id}"
    return save_pdf_png(fig, stem)


def plot_patient_trajectory_legacy(patient_id: int, bundle, output_dir: Path):
    """Historical points + smooth NODE lines, using the canonical component palette."""
    fit_w = bundle["fit_weights"]
    obs_t = bundle["observation_times"].copy()
    node_t = bundle["trajectory_times"].copy()
    traj = bundle["trajectory"]

    if LEGACY_INDIVIDUAL_NORMALIZED_TIME:
        # Historical presentation only: hides how much clinical time a
        # participant's observed support actually spans.
        lo, hi = float(node_t[0]), float(node_t[-1])
        span = hi - lo
        if span <= 0:
            raise ValueError("invalid individual NODE time support")
        node_x = (node_t - lo) / span
        obs_x = (obs_t - lo) / span
        xlabel = "Time (normalized, legacy presentation)"
        xlim = (-0.02, 1.02)
    else:
        # Clinical time. The NODE grid spans the observed support only, so the
        # drawn curve is interpolation, never extrapolation.
        node_x = node_t
        obs_x = obs_t
        xlabel = "Weeks since randomization"
        xlim = (float(node_t[0]), float(node_t[-1]))

    fig, ax = plt.subplots(figsize=(10, 5))
    for k in range(traj.shape[1]):
        color = COMPONENT_COLORS[k % len(COMPONENT_COLORS)]
        ax.scatter(obs_x, fit_w[:, k], s=18, alpha=0.55, color=color, edgecolors="none", zorder=3)
        ax.plot(node_x, traj[:, k], color=color, lw=2.4, zorder=4)

    ax.set_xlim(*xlim)
    ax.set_ylim(0.0, 0.8)
    ax.set_xlabel(xlabel, fontsize=15)
    ax.tick_params(labelsize=12)
    ax.grid(alpha=0.20, linestyle="--", linewidth=0.6)
    fig.tight_layout()
    return save_pdf_png(fig, output_dir / "trajectories")


def _patient_raw_blocks_d2(patient_id: int):
    data, metadata = load_patient_weeks(
        patient_id=int(patient_id),
        data_dir=str(PEDAP_DATA_DIR),
        representation="bivariate",
        return_metadata=True,
    )
    if patient_id not in data:
        raise RuntimeError(f"patient {patient_id}: raw d=2 blocks could not be reconstructed")
    return data[patient_id], metadata


def _gmm_density_2d_original(means, covs, weights, x_grid, y_grid):
    xx, yy = np.meshgrid(x_grid, y_grid)
    pos = np.dstack((xx, yy))
    density = np.zeros_like(xx, dtype=float)
    for weight, mean, covariance in zip(weights, means, covs):
        covariance = 0.5 * (covariance + covariance.T) + 1e-8 * np.eye(2)
        density += float(weight) * multivariate_normal(mean, covariance, allow_singular=True).pdf(pos)
    return xx, yy, density


def plot_patient_endpoint_d2(patient_id: int, bundle, raw_blocks, output_dir: Path, which: str):
    """Historical green GMM heatmap + black empirical cloud, no title/colorbar."""
    if which not in {"initial", "final"}:
        raise ValueError("which must be 'initial' or 'final'")

    index = 0 if which == "initial" else -1
    # SAP HIGH/LOW values (39 and 401 mg/dL) are part of the distribution and
    # are plotted; nothing is filtered out by a 40/400 rule here.
    empirical = np.asarray(raw_blocks[index], dtype=float)
    empirical = empirical[np.all(np.isfinite(empirical[:, :2]), axis=1)]

    obs_time = bundle["observation_times"][index]
    weights = node_weights_at_times(bundle, np.asarray([obs_time]))[0]

    x_grid = np.linspace(20.0, 400.0, 300)
    y_grid = np.linspace(-13.0, 19.0, 260)
    xx, yy, zz = _gmm_density_2d_original(
        bundle["means_original"], bundle["covs_original"], weights, x_grid, y_grid
    )

    fig, ax = plt.subplots(figsize=(10, 5.4))
    positive = zz[zz > 0]
    if positive.size:
        levels = np.linspace(0.0, float(np.quantile(positive, 0.995)), 120)
        levels = np.unique(levels)
        if len(levels) >= 2:
            ax.contourf(xx, yy, zz, levels=levels, cmap="Greens", alpha=0.48, extend="max")
    ax.scatter(
        empirical[:, 0], empirical[:, 1],
        s=6, color="black", alpha=0.28, linewidths=0, rasterized=True,
    )

    ax.set_xlim(20, 400)
    ax.set_ylim(-13, 19)
    ax.set_xlabel("Glucose concentration (mg/dL)", fontsize=15)
    ax.set_ylabel("Glucose rate of change (mg/dL/time)", fontsize=15)
    ax.tick_params(labelsize=12)
    ax.grid(alpha=0.17, linestyle="--", linewidth=0.6)
    fig.tight_layout()
    return save_pdf_png(fig, output_dir / which)


In [ ]:
def plot_patient_composite_d2(patient_id: int, bundle, raw_blocks, cgm: pd.DataFrame, output_dir: Path):
    """Four-panel manuscript convenience figure built from the same legacy-style ingredients."""
    arm = arm_for_patient(patient_id)
    arm_color = ARM_COLORS[arm_label_for_patient(patient_id)]
    patient_days = _patient_days(cgm, patient_id)

    fit_w = bundle["fit_weights"]
    obs_t = bundle["observation_times"]
    node_t = bundle["trajectory_times"]
    traj = bundle["trajectory"]
    # Clinical time: weeks since randomization.
    node_x = node_t
    obs_x = obs_t

    fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.0))

    # Raw daily CGM overlay.
    ax = axes[0, 0]
    for day in patient_days:
        m, g = _day_series(day)
        ax.plot(m / 5.0, g, color=arm_color, alpha=0.22, lw=0.5)
    ax.set(xlim=(0, 287), ylim=(0, 420), xlabel="Time of day (5-min slots)", ylabel="Glucose (mg/dL)")
    ax.grid(alpha=0.18)

    # MMD observations + NODE trajectory.
    ax = axes[0, 1]
    for k in range(traj.shape[1]):
        color = COMPONENT_COLORS[k % len(COMPONENT_COLORS)]
        ax.scatter(obs_x, fit_w[:, k], s=13, alpha=0.50, color=color, edgecolors="none")
        ax.plot(node_x, traj[:, k], lw=2.0, color=color)
    ax.set(xlim=(float(node_t[0]), float(node_t[-1])), ylim=(0, 0.8),
           xlabel="Weeks since randomization")
    ax.grid(alpha=0.18, linestyle="--", linewidth=0.6)

    for ax, which, index in ((axes[1, 0], "Initial", 0), (axes[1, 1], "Final", -1)):
        empirical = np.asarray(raw_blocks[index], dtype=float)
        empirical = empirical[np.all(np.isfinite(empirical[:, :2]), axis=1)]
        obs_time = bundle["observation_times"][index]
        weights = node_weights_at_times(bundle, np.asarray([obs_time]))[0]
        x_grid = np.linspace(20.0, 400.0, 250)
        y_grid = np.linspace(-13.0, 19.0, 220)
        xx, yy, zz = _gmm_density_2d_original(
            bundle["means_original"], bundle["covs_original"], weights, x_grid, y_grid
        )
        positive = zz[zz > 0]
        if positive.size:
            levels = np.unique(np.linspace(0.0, float(np.quantile(positive, 0.995)), 100))
            if len(levels) >= 2:
                ax.contourf(xx, yy, zz, levels=levels, cmap="Greens", alpha=0.46, extend="max")
        ax.scatter(empirical[:, 0], empirical[:, 1], s=4, color="black", alpha=0.25, linewidths=0, rasterized=True)
        ax.set(xlim=(20, 400), ylim=(-13, 19), xlabel="Glucose concentration (mg/dL)", ylabel="Glucose rate of change (mg/dL/time)")
        ax.set_title(which, fontsize=12)
        ax.grid(alpha=0.15, linestyle="--", linewidth=0.5)

    fig.tight_layout()
    return save_pdf_png(fig, output_dir / "composite")


In [ ]:
individual_outputs = {}
if DIMENSION == "d2":
    from load import load_pedap_cgm

    cgm_table = load_pedap_cgm(str(PEDAP_DATA_DIR))
    for patient_id in ROSENBLATT_IDS:
        output_dir = OUTPUT_FOLDER / "individual" / f"patient_{patient_id}"
        output_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n[individual] patient {patient_id} ({arm_for_patient(patient_id)})")
        bundle = load_patient_bundle(patient_id, RUN_FOLDER)
        raw_blocks, raw_metadata = _patient_raw_blocks_d2(patient_id)

        files = []
        files += plot_patient_timesignal(patient_id, cgm_table, output_dir)
        files += plot_patient_trajectory_legacy(patient_id, bundle, output_dir)
        files += plot_patient_endpoint_d2(patient_id, bundle, raw_blocks, output_dir, "initial")
        files += plot_patient_endpoint_d2(patient_id, bundle, raw_blocks, output_dir, "final")
        files += plot_patient_composite_d2(patient_id, bundle, raw_blocks, cgm_table, output_dir)

        individual_outputs[patient_id] = [str(path) for path in files]
        print(f" saved {len(files)} files -> {output_dir}")
else:
    print("[individual] skipped: current legacy endpoint/composite panels are d=2 only")


## 8. Optional per-week GMM-vs-empirical diagnostics for the selected run

In [ ]:
from Visualization.plots import plot_gmm_vs_empirical_1d, plot_gmm_vs_empirical_2d

GENERATE_PER_WEEK_DIAGNOSTICS = False

if GENERATE_PER_WEEK_DIAGNOSTICS:
    diagnostic_root = OUTPUT_FOLDER / "individual_weekly_diagnostics"
    for patient_id in ROSENBLATT_IDS:
        bundle = load_patient_bundle(patient_id, RUN_FOLDER)
        add_deriv = DIMENSION == "d2"
        blocks, block_metadata = load_patient_weeks(
            patient_id=int(patient_id),
            data_dir=str(PEDAP_DATA_DIR),
            representation=("bivariate" if add_deriv else "univariate"),
            return_metadata=True,
        )
        if patient_id not in blocks:
            print(f"[skip] patient {patient_id}: no usable weekly distribution")
            continue
        raw_blocks = blocks[patient_id]
        node_w = node_weights_at_times(bundle, bundle["observation_times"])
        patient_out = diagnostic_root / f"patient_{patient_id}"
        patient_out.mkdir(parents=True, exist_ok=True)

        # Files are named by the NOMINAL RCT week the block belongs to, so a
        # participant with gaps does not get its weeks silently renumbered.
        for week_index, block, weights in zip(
            block_metadata["nominal_week_indices"], raw_blocks, node_w
        ):
            save_path = patient_out / f"week_{week_index:02d}.png"
            if DIMENSION == "d1":
                plot_gmm_vs_empirical_1d(
                    bundle["means_original"], bundle["covs_original"], weights,
                    block, title=f"Patient {patient_id}, nominal week {week_index}", save_path=str(save_path)
                )
            else:
                plot_gmm_vs_empirical_2d(
                    bundle["means_original"], bundle["covs_original"], weights,
                    block, title=f"Patient {patient_id}, nominal week {week_index}", save_path=str(save_path)
                )


## 9. Figure inventory

All generated figures stay in `figures_paper/`. The table below lists the PDF files for the two active experiments; these files can be used directly in a manuscript.


In [ ]:
from Analysis.export_figures import write_inventory

inventory = write_inventory()
display(pd.DataFrame(inventory))
print("Inventory:", FIGURES_ROOT / "INVENTORY.md")


## 10. Reproducibility notes

- The notebook analyses exactly one selected run and one dimension at a time.
- `N_NOMINAL_WEEKS` in `Analysis/paper_config.py` is the trial's 13 nominal weekly periods, anchored at each participant's randomization date. It is the calendar structure of the RCT, not a requirement that every participant contribute 13 usable distributions.
- Loaders reject result files whose saved `dataset_fingerprint` does not match the processed PEDAP dataset, so clinical results are never mixed across datasets.
- Group figures use clinical time (weeks since randomization) on the union of observed supports, with available cases and arm-specific coverage counts: interpolation only, never extrapolation.
- The primary wild bootstrap uses one participant-level Rademacher multiplier for each complete vector-valued functional trajectory.
- Component-functional and pointwise tests are diagnostics and do not replace the primary global test.
- All outputs for the selected result set are written under `figures_paper/<dimension>/<experiment>/`.

- Functional inference uses a fixed cohort on the widest interval covered by at least 90% of each arm; 85% and 95% support-window sensitivities are reported separately.
